# Partie 1 : extraction d'entités dans BC5CDR (Chemical, Disease)

**Binôme** : AMARA Yanis, TOMAS Alban · Projet de NLP en santé (E5 BIO)

Dans cette première partie, nous présentons la tâche d'extraction NER (entités :  "Chemical" et "Disease") ainsi que la démarche sur les données, la façon de mesurer,
puis les deux méthodes (dictionnaire à règles, modèle appris), avec pour chacune les pistes essayées, et les résultats qu'on a pu obtenir.

**Tout est recalculé à l'exécution**, à partir de trois sources :
- le corpus d'origine `CDR_Data/` (fichiers PubTator de BC5CDR, et prédictions des outils officiels tmChem / DNorm fournis avec le dataset) ;
- le lexique médical **MeSH 2026** de la NLM (téléchargé automatiquement dans `data/mesh/` à la première exécution, ≈ 64 Mo) ;
- le code partagé `src/` (dictionnaire et règles, préparation BIO, prédiction) et `tals/` (cours) ;
- le modèle ajusté **APP-v3** (`modeles/apprise_v3`, PubMedBERT ajusté sur Google Colab), rechargé pour prédire.

Les entraînements des modèles BERT ne sont pas refaits (GPU nécessaire) : leurs courbes de réglage sont celles
enregistrées pendant l'entraînement. Une cellule optionnelle (section 5.5) permet de réentraîner APP-v3 sur GPU.

Les prédictions TEST sont écrites dans `rapport/sorties/`.

In [1]:
import sys, json, time, random, warnings
from collections import Counter
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from tals import Mention, erreurs, tokeniser_avec_offsets, mentions_vers_bio, bio_vers_mentions
from src.cdr import CATEGORIES, charger_cdr, mentions_reference, decouper_phrases
from src.dictionnaire import DictionnaireCDR, ConfigDictionnaire
from src.dictionnaire_v3 import DictionnaireV3, REGLES_V2
from src.dictionnaire_v4 import DictionnaireV4
from src.mesh import charger_lexique
from src.extraction import tableau_scores, ecrire_predictions

CDR = RACINE / "CDR_Data"
HISTORIQUE = RACINE / "resultats" / "extraction"    # courbes enregistrées pendant les entraînements (Colab)
SORTIES = RACINE / "rapport" / "sorties"
SORTIES.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

parts = charger_cdr(CDR / "CDR.Corpus.v010516")        # TRAIN / VALIDATION / TEST officiels
ref = {p: mentions_reference(docs) for p, docs in parts.items()}
textes = {d.document_id: d.texte for docs in parts.values() for d in docs}
print({p: len(docs) for p, docs in parts.items()}, "documents")

{'train': 500, 'validation': 500, 'test': 500} documents


## 1. Repérage des mentions

Nous reprérons dans un résumé d'article médical (PubMed), chaque **mention** d'un produit chimique ou médicament
(`Chemical`) et d'une maladie ou d'un symptôme (`Disease`), avec sa position exacte dans le texte (début, fin) et
son type. Une mention est un fragment de texte : « cocaine », « myocardial infarction », « 5-FU ».

Exemple sur le plus court résumé de VALIDATION, mentions de référence entre crochets :

In [2]:
def surligner(texte, mentions):
    """Texte avec chaque mention entre [crochets|Type]."""
    sortie, pos = [], 0
    for m in sorted(mentions, key=lambda m: m.debut):
        sortie += [texte[pos:m.debut], f"[{texte[m.debut:m.fin]}|{m.type[:4]}]"]
        pos = m.fin
    return "".join(sortie + [texte[pos:]])

exemple = min(parts["validation"], key=lambda d: len(d.texte))
print(exemple.document_id, "—", len(exemple.mentions), "mentions\n")
print(surligner(exemple.texte, mentions_reference([exemple])))

10565806 — 5 mentions

[Hypomania|Dise]-like syndrome induced by [olanzapine|Chem]. We report a female patient with a diagnosis of a not otherwise specified [psychotic disorder|Dise] (DSM-IV) who developed [hypomania|Dise] shortly after the introduction of [olanzapine|Chem] treatment.


## 2. Le corpus d'extraction : BC5CDR

Corpus **BC5CDR** (BioCreative V, 2015) : 1 500 résumés PubMed annotés par des experts en `Chemical` et `Disease`,
avec trois partitions officielles TRAIN, VALIDATION et TEST que nous gardons telles quelles.

Nous l'avons choisi parce qu'il est annoté avec soin (accord entre annotateurs
≈ 0,89), qu'il est déjà partitionné et qu'il sert de référence dans les publications : scores obtenus comparables à l'état de l'art (section 7). Le texte d'un document est le titre suivi du résumé ; les positions sont
en caractères. 

On vérifie d'abord qu'aucun document n'apparaît dans deux partitions :

In [3]:
noms = list(parts)
for i, a in enumerate(noms):
    for b in noms[i + 1:]:
        communs = {d.document_id for d in parts[a]} & {d.document_id for d in parts[b]}
        print(f"{a} ∩ {b} : {len(communs)} document commun")

lignes = []
for p, docs in parts.items():
    ms = [m for d in docs for m in d.mentions]
    lignes.append({"partition": p, "documents": len(docs),
                   "Chemical": sum(m.type == "Chemical" for m in ms), "Disease": sum(m.type == "Disease" for m in ms),
                   "mentions": len(ms), "formes distinctes": len({m.texte.lower() for m in ms}),
                   "composites": sum(m.composite for m in ms),
                   "mots par mention (moy.)": round(sum(len(m.texte.split()) for m in ms) / len(ms), 2)})
pd.DataFrame(lignes).set_index("partition")

train ∩ validation : 0 document commun
train ∩ test : 0 document commun
validation ∩ test : 0 document commun


,documents,Chemical,Disease,mentions,formes distinctes,composites,mots par mention (moy.)
partition,,,,,,,
train,500,5203,4182,9385,2390,107,1.33
validation,500,5347,4244,9591,2235,89,1.31
test,500,5385,4424,9809,2355,57,1.29


On peut remarquer que les trois partitions sont de taille et de composition très proches (≈ 55 % Chemical, 45 % Disease). 

Deux points ont compté pour la suite : 

**Mentions composites** (« ovarian and testicular cancer ») : une seule mention qui en contient deux. Nous les
gardons entières, comme dans la référence.

**Conversion en étiquettes mot par mot (BIO)** : Cette étape est nécessaire pour la méthode apprise ==> chaque mot (ou signe de ponctuation)
reçoit `B-<type>` (début de mention), `I-<type>` (suite) ou `O` (hors mention). 

On vérifie qu'en reconvertissant les
étiquettes en mentions on retrouve bien la référence :

In [4]:
bio, bilan = {}, []
for p, docs in parts.items():
    perdues = hors_token = 0
    for d in docs:
        mentions = {Mention(d.document_id, m.debut, m.fin, m.type) for m in d.mentions}
        tokens = tokeniser_avec_offsets(d.texte)
        offsets = [(a, b) for _, a, b in tokens]
        debuts, fins = {a for a, _ in offsets}, {b for _, b in offsets}
        hors_token += sum(m.debut not in debuts or m.fin not in fins for m in mentions)
        tags = mentions_vers_bio(mentions, offsets)
        perdues += len(mentions - bio_vers_mentions(tags, offsets, d.document_id, politique="stricte"))
        bio[d.document_id] = {"document_id": d.document_id, "tokens": [s for s, _, _ in tokens], "offsets": offsets, "tags": tags}
    bilan.append({"partition": p, "mentions": len(ref[p]), "bornes au milieu d'un mot": hors_token,
                  "perdues à l'aller-retour BIO": perdues})
display(pd.DataFrame(bilan).set_index("partition"))
b = bio[parts["train"][0].document_id]
pd.DataFrame({"mot": b["tokens"][:14], "étiquette": b["tags"][:14]}).T

,mentions,bornes au milieu d'un mot,perdues à l'aller-retour BIO
partition,,,
train,9385,0,0
validation,9591,10,10
test,9809,5,5


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
mot,Naloxone,reverses,the,antihypertensive,effect,of,clonidine,.,In,unanesthetized,",",spontaneously,hypertensive,rats
étiquette,B-Chemical,O,O,O,O,O,B-Chemical,O,O,O,O,O,B-Disease,O


Quelques mentions de VALIDATION et TEST commencent ou finissent au milieu d'un mot (« 5-HT » dans « 5-HT6 ») : le
format BIO ne peut pas les représenter, elles resteront comptées comme erreurs pour le modèle appris.

Une première limite à la méthode par dictionnaire construit sur TRAIN est qu'une partie non négligeable de mentions apparaît dans VALIDATION et TEST sans jamais être annotée dans TRAIN. Nous vérifions à quelle proportion elle s'élève.

In [5]:
formes_train = {m.texte.lower() for d in parts["train"] for m in d.mentions}
for p in ("validation", "test"):
    ms = [m for d in parts[p] for m in d.mentions]
    inconnues = [m for m in ms if m.texte.lower() not in formes_train]
    par_type = Counter(m.type for m in inconnues)
    print(f"{p:10s} : {len(inconnues)} mentions sur {len(ms)} ({len(inconnues) / len(ms):.1%}) ont une forme absente de TRAIN"
          f" — Chemical {par_type['Chemical']}, Disease {par_type['Disease']}")

validation : 3500 mentions sur 9591 (36.5%) ont une forme absente de TRAIN — Chemical 1897, Disease 1603
test       : 3703 mentions sur 9809 (37.8%) ont une forme absente de TRAIN — Chemical 2088, Disease 1615


Plus d'un tiers des mentions (36 à 38 %) ont une forme jamais annotée dans TRAIN. Une simple recherche des formes de
TRAIN ne peut donc pas dépasser ≈ 63 % de rappel. Pour pallier ce problème, il faut ajouter des règles au dictionnaire qui reconnaissent des
formes nouvelles (section 4) ou un modèle capable de généraliser (section 5).

## 3. Méthode de mesure

Pour garantir une correspondance exacte, on considère qu'une prédiction est juste si le document, le début, la fin et le type coïncident
  avec la référence. Par exemple :  « infarction » est différent de « myocardial infarction » et compte comme une erreur double
  (un faux positif et un faux négatif).

Pour l'évaluation des prédictions, on utilise les métriques habituelles : 
- **Précision** P = VP / (VP + FP) : part des prédictions justes.
- **Rappel** R = VP / (VP + FN) : part des mentions de référence retrouvées.
- **F1** = moyenne harmonique de P et R.
- **Micro-moyenne** : on additionne les comptes des deux catégories avant de calculer les ratios.

Nous utilisons dans la suite et pour les deux méthodes le scoreur (`src/extraction.py`) qui repose sur le code `tals.evaluation` du cours.
Nous nous sommes assurés que tous les réglages étaient choisis sur VALIDATION.
La partition TEST n'a été utilisé qu'à la fin, sur les versions figées.

Nous notons dans la suite que le score sur VALIDATION des versions choisies est souvent un peu optimiste comparé à l'évaluation sur l'ensemble TEST, ce qui est attendu.

In [6]:
def scores(pred, partition="validation", nom=""):
    """P, R, F1 global et F1 par catégorie."""
    s = tableau_scores(ref[partition], pred, nom, partition).set_index("categorie")
    return {"P": s.loc["global (micro)", "precision"], "R": s.loc["global (micro)", "rappel"],
            "F1": s.loc["global (micro)", "F1"], "F1 Chemical": s.loc["Chemical", "F1"], "F1 Disease": s.loc["Disease", "F1"]}

## 4. Méthode 1 : dictionnaire et règles

### 4.1 Le principe (DIC-v1)

Pour cette première version du dictionnaire à règles, nous listons toutes les mentions annotées de TRAIN avec leur type (« cocaine » → Chemical). 
Dans un nouveau texte, on cherche ces formes en prenant la plus longue correspondance à chaque position (ex: « myocardial infarction » plutôt que « infarction »), sans chevauchement. 
Nous utilisons pour cela le code (`src/dictionnaire.py`) basé sur l'extracteur du TD, avec deux corrections :

- les formes sont découpées avec le même tokeniseur que le texte (permet de retrouver les mentions du type « 5-FU » qui ne l'était jamais)
- une forme annotée avec les deux types reçoit son type majoritaire.

Nous avons dans un premier temps envisagé trois choix que nous avons réglés ensuite sur VALIDATION (16 combinaisons) :
- **casse** exacte ou insensible (« Cocaine » = « cocaine ») ;
- **seuil de taux d'annotation** : une forme n'est gardée que si, dans TRAIN, elle est annotée au moins dans cette
  proportion de ses apparitions (par exemple « drug » apparaît souvent sans être annoté, donc le garder créerait des faux positifs) 
- **expressions régulières de suffixes** (*-mycin, -olol, -pril* pour "Chemical") ;
  *-itis, -emia, -pathy* pour "Disease"), cette méthode permettrait d'attraper des formes absentes de TRAIN.

In [7]:
reglage = []
for casse in ("exacte", "insensible"):
    for seuil in (0.0, 0.3, 0.5, 0.7):
        for regex in (False, True):
            t0 = time.perf_counter()
            dic = DictionnaireCDR(ConfigDictionnaire(casse, seuil, regex)).construire(parts["train"])
            pred = dic.extraire_corpus(parts["validation"])
            reglage.append({"casse": casse, "seuil": seuil, "regex": regex, "entrées": len(dic.entrees),
                            **scores(pred), "temps (s)": round(time.perf_counter() - t0, 1)})
reglage = pd.DataFrame(reglage).sort_values(["F1", "P"], ascending=False)
reglage

,casse,seuil,regex,entrées,P,R,F1,F1 Chemical,F1 Disease,temps (s)
11,insensible,0.3,True,2348,0.8835,0.6438,0.7449,0.7715,0.7127,2.5
13,insensible,0.5,True,2319,0.8876,0.6340,0.7397,0.7697,0.7029,2.5
1,exacte,0.0,True,2719,0.8675,0.6354,0.7335,0.7554,0.7073,2.9
3,exacte,0.3,True,2683,0.8818,0.6268,0.7328,0.7582,0.7020,2.8
5,exacte,0.5,True,2658,0.8862,0.6180,0.7282,0.7566,0.6934,2.8
10,insensible,0.3,False,2348,0.8903,0.6153,0.7277,0.7557,0.6934,2.5
12,insensible,0.5,False,2319,0.8954,0.6041,0.7215,0.7538,0.6811,2.5
15,insensible,0.7,True,2229,0.8898,0.6055,0.7206,0.7639,0.6663,2.8
0,exacte,0.0,False,2719,0.8690,0.6045,0.7130,0.7352,0.6864,3.2
2,exacte,0.3,False,2683,0.8881,0.5941,0.7119,0.7378,0.6803,2.7


**Lecture et décision** : la meilleure combinaison est **casse insensible, seuil 0,3, regex** (F1 0,745). 

Sans seuil, la casse insensible fait *baisser* le score. En cherchant pourquoi, nous avons trouvé que quelques formes
courtes devenaient ambiguës quand on ignorait la casse

In [8]:
base = DictionnaireCDR(ConfigDictionnaire("insensible", 0.0, False)).construire(parts["train"])
stats = pd.DataFrame([{"forme": " ".join(k), **v} for k, v in base.stats.items()])
stats.query("occurrences >= 20").sort_values("taux").head(8)

,forme,type,annotations,occurrences,taux
1049,mg,Chemical,3,386,0.007772
2133,be,Chemical,7,327,0.021407
564,all,Disease,4,129,0.031008
2151,h,Chemical,3,79,0.037975
471,can,Disease,3,71,0.042254
221,infarction,Disease,4,48,0.083333
782,ng,Chemical,3,26,0.115385
425,no,Chemical,25,186,0.134409


On peut voir par exemple que « mg » en casse insensible peut être **Mg** (magnésium) annoté 3 fois, mais aussi et surtout l'unité, d'où les 386 occurrences pour 3 annotations seulement.

« all » peut être **ALL** mentionné comme une leucémie, mais surtout le mot anglais).
De même que pour « be » ou « can » qui apparaissent des centaines de fois sans être des mentions également. 

Nous avons donc introduit un seuil sur le taux d'annotation pour corriger les effets
de la casse insensible. 

Nous avons remarqué un autre biais sur DIC-v1: « infarction », compté comme « non annoté » quand il fait
partie de « myocardial infarction », était retiré à tort. Ce point a été corrigé plus tard (DIC-v3, ajout A).


In [9]:
pred_v1 = DictionnaireCDR(ConfigDictionnaire("insensible", 0.3, True)).construire(parts["train"]).extraire_corpus(parts["validation"])
tableau_scores(ref["validation"], pred_v1, "DIC-v1", "validation").set_index("categorie").round(3)

,methode,partition,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
categorie,,,,,,,,,,
Chemical,DIC-v1,validation,3501,228,1846,0.939,0.655,0.772,False,None
Disease,DIC-v1,validation,2674,586,1570,0.820,0.630,0.713,False,None
global (micro),DIC-v1,validation,6175,814,3416,0.884,0.644,0.745,False,None


L'évaluation de DIC-V1 est très informative :
- **précision élevée (0,88)**
- **rappel faible (0,64)**

Nous pouvons noter que le rappel est proche du plafond (≈ 63 %) calculé en section 2.
Le dictionnaire se trompe rarement sur ce qu'il trouve, mais il manque beaucoup de
mentions. La suite consiste donc à retrouver des mentions manquées sans abîmer la précision.

### 4.2 Des règles à partir des erreurs (DIC-v2)

Nous avons lu les mentions manquées de VALIDATION et cherché des régularités, puis écrit une règle pour chacune, en
l'**apprenant sur TRAIN** quand c'était possible (et non à la main), pour ne pas sur-ajuster sur VALIDATION. Code :
`src/dictionnaire_v3.py`.

| Règle | Idée | Comment elle est construite |
|---|---|---|
| Suffixes appris | *-dipine, -anyl, -olam* → Chemical ; *-lemia, -bosis* → Disease | suffixes vus ≥ 8 fois dans TRAIN, annotés ≥ 80 % du temps |
| Modificateurs à gauche | « tardive », « myocardial », « postoperative » allongent une maladie vers la gauche | mots inclus ≥ 70 % du temps dans l'annotation, sur TRAIN |
| « X-induced » | dans « cocaine-induced », X est presque toujours Chemical | 82 % dans TRAIN ; liste d'exclusion (mots génériques) |
| Sigles définis | « combined oral contraceptives (COCs) » : le sigle prend le type de la forme longue | principe de Schwartz & Hearst (2003) |
| Préfixe anti- | « antihypertensive » n'est presque jamais annoté | retrait des prédictions en *anti-* |

Effet mesuré sur VALIDATION, chaque règle seule puis toutes ensemble :

In [10]:
def dic(**options):
    return DictionnaireV3(**{"filtre_corrige": False, "variantes": False, "extension": False, "propagation": False,
                             **options}).construire(parts["train"])

dic_v1 = dic(regles=())
pred_v1 = dic_v1.extraire_corpus(parts["validation"])
effet_regles = [{"règles": "aucune (DIC-v1)", **scores(pred_v1)}]
for r in REGLES_V2:
    p = dic(regles=(r,)).extraire_corpus(parts["validation"])
    effet_regles.append({"règles": f"+ {r} seule", **scores(p), "ajouts justes": len((p - pred_v1) & ref["validation"]),
                         "ajouts faux": len((p - pred_v1) - ref["validation"])})
dic_v2 = dic()
pred_v2 = dic_v2.extraire_corpus(parts["validation"])
effet_regles.append({"règles": "les cinq, dans l'ordre (DIC-v2)", **scores(pred_v2),
                     "ajouts justes": len((pred_v2 - pred_v1) & ref["validation"]),
                     "ajouts faux": len((pred_v2 - pred_v1) - ref["validation"])})
print(f"Appris sur TRAIN : {sum(len(m.pattern.split('|')) for m in dic_v2.regex_suffixes.values())} suffixes, "
      f"{len(dic_v2.modificateurs)} modificateurs à gauche (ex. {sorted(dic_v2.modificateurs)[:6]})")
pd.DataFrame(effet_regles).set_index("règles")

Appris sur TRAIN : 108 suffixes, 49 modificateurs à gauche (ex. ['abdominal', 'absence', 'allograft', 'aplastic', 'auditory', 'cardiac'])


,P,R,F1,F1 Chemical,F1 Disease,ajouts justes,ajouts faux
règles,,,,,,,
aucune (DIC-v1),0.8835,0.6438,0.7449,0.7715,0.7127,NaN,NaN
+ suffixes seule,0.8814,0.6696,0.7610,0.7997,0.7128,247.0,50.0
+ modificateurs seule,0.8901,0.6486,0.7504,0.7715,0.7249,64.0,26.0
+ induit seule,0.8814,0.6495,0.7479,0.7767,0.7127,54.0,24.0
+ sigles seule,0.8846,0.6744,0.7653,0.7935,0.7310,293.0,35.0
+ anti seule,0.8857,0.6438,0.7456,0.7728,0.7129,0.0,0.0
"les cinq, dans l'ordre (DIC-v2)",0.8889,0.7128,0.7912,0.8289,0.7434,679.0,134.0


On peut remarquer que chaque règle augmente un peu plus le rappel, ensemble, elles portent le F1 de 0,745 à **0,791** (DIC-v2), le rappel de
0,64 à 0,71 tout en conservant une précision relativement stable (0,89).

Les sigles et les suffixes appris sont les plus utiles. Nous avons
aussi essayé et **abandonné** deux idées : 
   - **Une regex qui attrappe les sigles en lettres capitales**  
   - **Une regex qui attrape les formes de deux lettres "Xx" (surtout pour les symboles chimiques)**

Nous les avons mesurés sur TRAIN avant de les appliquer :

In [11]:
import re
def precision_sur_train(motif, filtre=lambda t, a: True):
    vus = justes = 0
    for d in parts["train"]:
        annot = {(m.debut, m.fin) for m in d.mentions}
        for r in re.finditer(motif, d.texte):
            if filtre(d.texte, r.start()):
                vus += 1
                justes += r.span() in annot
    return {"occurrences dans TRAIN": vus, "annotées (précision)": f"{justes / vus:.0%}"}

pd.DataFrame({
    "sigle en capitales [A-Z]{2,6}": precision_sur_train(r"\b[A-Z]{2,6}\b"),
    "deux lettres « Xx » hors début de phrase": precision_sur_train(r"\b[A-Z][a-z]\b",
                                                                    lambda t, i: i > 1 and t[i - 2] not in ".!?"),
}).T

,occurrences dans TRAIN,annotées (précision)
"sigle en capitales [A-Z]{2,6}",2219,37%
deux lettres « Xx » hors début de phrase,178,29%


Une regex générique pour les sigles attrape *II, CI, HIV, DNA*… 
Les formes de deux lettres attrapent *To, We, In*,
« Hg » de « mm Hg »

En conclusion, elles capturent trop de faux positifs pour être utiles. De plus, une liste de symboles chimiques *Na, K, Ca* n'apportait
rien, étant déjà dans le dictionnaire.

### 4.3 Dernière analyse : ce qui reste atteignable (DIC-v3)

Avant d'aller plus loin, nous avons fais le bilan des mentions encore manquées par DIC-v2 :

In [12]:
err_v2 = erreurs(ref["validation"], pred_v2)
manquees_v2 = err_v2["manquee"]
forme = lambda m: textes[m.doc][m.debut:m.fin]
inconnue = [m for m in manquees_v2 if forme(m).lower() not in formes_train]
sigles = [m for m in inconnue if re.fullmatch(r"[A-Z0-9][A-Za-z0-9-]{1,9}", forme(m)) and any(c.isupper() for c in forme(m)[1:])]
connues = [m for m in manquees_v2 if forme(m).lower() in formes_train]
ailleurs = [m for m in manquees_v2 if any(forme(x) == forme(m) for x in pred_v2 if x.doc == m.doc)]
pd.Series({"mentions manquées par DIC-v2": len(manquees_v2),
           "dont forme jamais vue dans TRAIN": len(inconnue),
           "    dont sigles": len(sigles),
           "dont forme vue dans TRAIN (retirée par le filtre, autre casse, autre type…)": len(connues),
           "dont forme reconnue ailleurs dans le même résumé": len(ailleurs),
           "frontières fausses (bonne zone, mauvaises bornes)": len(err_v2["frontiere"])}).to_frame("nombre")

,nombre
mentions manquées par DIC-v2,2171
dont forme jamais vue dans TRAIN,2065
dont sigles,383
"dont forme vue dans TRAIN (retirée par le filtre, autre casse, autre type…)",106
dont forme reconnue ailleurs dans le même résumé,83
"frontières fausses (bonne zone, mauvaises bornes)",576


La grande majorité des manques sont des formes jamais vues, hors de portée d'un dictionnaire. 

Pour les quelques familles corrigeables, nous avons essayé de les traiter par quatre ajouts.
Nous avons décidé de conserver un ajout si et seulement si le garder améliore seul le F1 VALIDATION.

| Ajout | Idée |
|---|---|
| A — filtre corrigé | une occurrence incluse dans une mention plus longue ne compte plus comme « non annotée » (*cancer, edema*) |
| B — variantes singulier/pluriel | *tumor* ↔ *tumors*, ajoutée seulement si la variante n'apparaît jamais dans TRAIN |
| C — extension à droite | suites de 1 à 3 mots qui prolongent une mention dans TRAIN (≥ 3 fois, ≥ 70 %), symétrique des modificateurs |
| D — propagation | une forme reconnue une fois dans un résumé est reconnue à ses autres occurrences (même casse) |

In [13]:
AJOUTS = {"A — filtre corrigé": {"filtre_corrige": True}, "B — variantes": {"variantes": True},
          "C — extension à droite": {"extension": True}, "D — propagation": {"propagation": True}}
effet_ajouts = [{"version": "DIC-v2", **scores(pred_v2)}]
for nom, opt in AJOUTS.items():
    p = dic(**opt).extraire_corpus(parts["validation"])
    effet_ajouts.append({"version": f"+ {nom}", **scores(p), "ajouts justes": len((p - pred_v2) & ref["validation"]),
                         "ajouts faux": len((p - pred_v2) - ref["validation"])})
effet_ajouts = pd.DataFrame(effet_ajouts).set_index("version")
gardes = [n for n in AJOUTS if effet_ajouts.loc[f"+ {n}", "F1"] > effet_ajouts.loc["DIC-v2", "F1"]]
print("Ajouts qui améliorent seuls le F1 :", [n[0] for n in gardes])

t0 = time.perf_counter()
dic_v3 = DictionnaireV3(**{k: True for n in gardes for k in AJOUTS[n]},
                        **{k: False for n in AJOUTS if n not in gardes for k in AJOUTS[n]}).construire(parts["train"])
t_construction = time.perf_counter() - t0
t0 = time.perf_counter()
pred_v3 = dic_v3.extraire_corpus(parts["validation"])
t_extraction = time.perf_counter() - t0
effet_ajouts.loc["DIC-v3 (ajouts gardés)"] = {**scores(pred_v3), "ajouts justes": len((pred_v3 - pred_v2) & ref["validation"]),
                                              "ajouts faux": len((pred_v3 - pred_v2) - ref["validation"])}
print(f"Suites apprises pour l'extension à droite : {len(dic_v3.suites)} ; variantes ajoutées : {dic_v3.variantes_ajoutees}")
effet_ajouts

Ajouts qui améliorent seuls le F1 : ['A', 'B', 'C', 'D']


Suites apprises pour l'extension à droite : 2 ; variantes ajoutées : 1928


,P,R,F1,F1 Chemical,F1 Disease,ajouts justes,ajouts faux
version,,,,,,,
DIC-v2,0.8889,0.7128,0.7912,0.8289,0.7434,NaN,NaN
+ A — filtre corrigé,0.8841,0.7214,0.7945,0.8293,0.7513,83.0,57.0
+ B — variantes,0.8883,0.7180,0.7941,0.8298,0.7493,51.0,16.0
+ C — extension à droite,0.8892,0.7130,0.7914,0.8289,0.7440,2.0,2.0
+ D — propagation,0.8866,0.7174,0.7931,0.8321,0.7434,45.0,26.0
DIC-v3 (ajouts gardés),0.8811,0.7314,0.7993,0.8333,0.7569,180.0,101.0


Gain de **+0,8 point** seulement (0,791 → 0,799), inférieur à ce que nous espérions (+2 à 3 points).
L'extension à droite n'a trouvé que deux suites fréquentes dans TRAIN (son effet est nul, gardé quand même étant donné la légère augmentation de F1). Le filtre corrigé réintroduit « infection » ou « cancer » y compris là où ils font partie d'une mention
plus longue (frontières fausses).
L'ajout de ces règles permet néanmoins plus d'ajouts justes que faux, ce qui permet cette légère augmentation du score. 

Ce qui reste manqué par DIC-v3 :

In [14]:
manquees_v3 = erreurs(ref["validation"], pred_v3)["manquee"]
hors_train = [m for m in manquees_v3 if forme(m).lower() not in formes_train]
print(f"DIC-v3 : {len(manquees_v3)} mentions manquées, dont {len(hors_train)} ({len(hors_train) / len(manquees_v3):.0%}) "
      "ont une forme absente de TRAIN")
print("\nExemples :", sorted({forme(m) for m in random.Random(42).sample(hors_train, 12)}))

DIC-v3 : 1941 mentions manquées, dont 1907 (98%) ont une forme absente de TRAIN

Exemples : ['Ammonia', 'CLZ', 'Dexrazoxane', 'LSD', 'MDs', 'QTC prolongation', 'adenovirus disease', 'decreased renal function', 'dimethyl sulphoxide', 'thiol', 'tic-like symptoms']


**Bilan de DIC-v3** : presque toutes les mentions encore manquées ont une forme absente de TRAIN. 

Le F1 de VALIDATION est de 0,799, la précision reste forte, mais le rappel est toujours limité par le vocabulaire. 

Pour pallier ce problème de vocabulaire et nous plaçer dans les mêmes conditions que les autres modèles travaillant sur les données de PubMed, dont le modèle de référence fournit avec ce dataset (DNorm/tmChem), nous testons par la suite l'ajout d'un lexique externe à notre vocabulaire.


### 4.4 Un lexique externe : MeSH (DIC-v4)

**MeSH** (*Medical Subject Headings*, NLM) est le vocabulaire médical de PubMed ; c'est aussi celui qui sert à
normaliser les annotations de BC5CDR. Nous en reprenons les maladies (arbres C et F03 → Disease), les produits
chimiques et médicaments (arbre D → Chemical), les fiches supplémentaires de substances et de maladies rares, avec tous
leurs synonymes : environ 840 000 termes (`src/mesh.py`). Ils sont ajoutés au dictionnaire de DIC-v3, et les règles
restent les mêmes (`src/dictionnaire_v4.py`).

Ajouter tel quel un tel lexique crée beaucoup de faux positifs (*drug*, *protein*, *enzyme* sont des termes MeSH).

Nous ajoutons trois garde-fous, tous **appris sur TRAIN** avec le seuil de 30 % déjà utilisé :
- **priorité à TRAIN** : une forme annotée dans TRAIN garde la décision de DIC-v3 ;
- **filtre par branche** : MeSH est un arbre (ex. D12.776 = protéines). Une branche dont les termes, cherchés dans
  TRAIN, sont annotés moins de 30 % du temps est écartée : BC5CDR n'annote ni les protéines, ni les enzymes, ni les
  classes de médicaments (*analgesic*) ;
- **filtre par terme** : un terme MeSH trouvé dans TRAIN mais rarement annoté est retiré (*lead*, *water*).

Variantes comparées sur VALIDATION ; on garde celle qui a le meilleur F1 :

In [15]:
lexique = charger_lexique(RACINE / "data" / "mesh")       # téléchargé et mis en cache à la première exécution
VARIANTES_MESH = {"MeSH sans filtre": dict(niveau_branches=0, filtre_termes=False),
                  "+ filtre par terme": dict(niveau_branches=0),
                  "+ filtre par branche (1er niveau, « D12 »)": dict(niveau_branches=1),
                  "+ filtre par branche (2e niveau, « D12.776 »)": dict(niveau_branches=2),
                  "2e niveau, sans les fiches supplémentaires": dict(niveau_branches=2, sources=("descripteur",))}
effet_mesh = [{"version": "DIC-v3", **scores(pred_v3)}]
for nom, opt in VARIANTES_MESH.items():
    d = DictionnaireV4(lexique, **opt).construire(parts["train"])
    p = d.extraire_corpus(parts["validation"])
    effet_mesh.append({"version": nom, **scores(p), "termes MeSH ajoutés": len(d.mesh_ajoutees),
                       "ajouts justes": len((p - pred_v3) & ref["validation"]), "ajouts faux": len((p - pred_v3) - ref["validation"])})
    del d
effet_mesh = pd.DataFrame(effet_mesh).set_index("version")
choix_mesh = effet_mesh.iloc[1:]["F1"].idxmax()
print("Variante retenue :", choix_mesh)

t0 = time.perf_counter()
dic_v4 = DictionnaireV4(lexique, **VARIANTES_MESH[choix_mesh]).construire(parts["train"])
t_construction_v4 = time.perf_counter() - t0
t0 = time.perf_counter()
pred_v4 = dic_v4.extraire_corpus(parts["validation"])
t_extraction_v4 = time.perf_counter() - t0
print(f"{len(dic_v4.branches_ecartees)} branches écartées, dont :",
      sorted(dic_v4.branches_ecartees & {"D12.776", "D08.811", "D26", "D27.505"}))
effet_mesh

Variante retenue : + filtre par branche (2e niveau, « D12.776 »)


29 branches écartées, dont : ['D08.811', 'D12.776', 'D26', 'D27.505']


,P,R,F1,F1 Chemical,F1 Disease,termes MeSH ajoutés,ajouts justes,ajouts faux
version,,,,,,,,
DIC-v3,0.8811,0.7314,0.7993,0.8333,0.7569,NaN,NaN,NaN
MeSH sans filtre,0.7733,0.8598,0.8142,0.8054,0.8268,824542.0,1422.0,1828.0
+ filtre par terme,0.8580,0.8633,0.8607,0.8740,0.8426,823925.0,1377.0,771.0
"+ filtre par branche (1er niveau, « D12 »)",0.8823,0.8627,0.8724,0.8954,0.8421,692858.0,1332.0,486.0
"+ filtre par branche (2e niveau, « D12.776 »)",0.9032,0.8669,0.8847,0.9174,0.8424,443740.0,1345.0,270.0
"2e niveau, sans les fiches supplémentaires",0.9017,0.8458,0.8729,0.8984,0.8405,93209.0,1141.0,240.0


- Sans filtre, le rappel augmente de beaucoup (0,73 → 0,86) mais la précision chute de manière importante (0,77) : MeSH contient des mots courants.
- Chaque garde-fou fait remonter la précision ; le filtre par branche au 2e niveau la porte **au-dessus de
  DIC-v3** (0,90 contre 0,88), en gardant le gain de rappel.
- **DIC-v4** gagne **+8,5 points** (0,799 → 0,885), autant en Chemical qu'en Disease : c'est le plus gros gain de la
  méthode 1.

   Les fiches supplémentaires (noms de molécules rares, codes comme *SM-5887*) apportent à elles seules +1,2 point.

Ce que MeSH apporte, et ce qu'il introduit :

In [16]:
def plus_frequentes(mentions, n):
    return [f"{f} ({k})" for f, k in Counter(forme(m) for m in mentions).most_common(n)]

groupes = {"ajouts justes": ((pred_v4 - pred_v3) & ref["validation"], 10),
           "ajouts faux": ((pred_v4 - pred_v3) - ref["validation"], 10),
           "pertes": ((pred_v3 - pred_v4) & ref["validation"], 6)}
bilan_mesh = pd.DataFrame({f"{nom} ({len(m)} au total)": pd.Series(plus_frequentes(m, n))
                           for nom, (m, n) in groupes.items()}).fillna("")
bilan_mesh.index += 1

manquees_v4 = erreurs(ref["validation"], pred_v4)["manquee"]
display(pd.DataFrame({"DIC-v3": [len(manquees_v3)], "DIC-v4": [len(manquees_v4)]}, index=["mentions manquées"]))
bilan_mesh

,DIC-v3,DIC-v4
mentions manquées,1941,832


,ajouts justes (1345 au total),ajouts faux (270 au total),pertes (46 au total)
1,iohexol (14),gellan (8),hypertension (4)
2,remifentanil (12),PTH (8),seizures (4)
3,cibenzoline (11),DOC (5),iron (3)
4,CB 3717 (11),tic (5),urticaria (3)
5,ammonia (11),surface antigen (5),pain (2)
6,suprofen (10),glucagon (5),sodium (2)
7,torsade de pointes (10),essential hypertension (4),
8,optic neuropathy (10),PVC (4),
9,vitamin C (10),zymosan (3),
10,uric acid (10),mineral (3),


Les faux positifs restants sont surtout des hormones et sigles que BC5CDR n'annote pas (*PTH*, *glucagon*), et
quelques mentions sont perdues quand MeSH propose une forme plus longue que l'annotation (*essential hypertension* au
lieu de *hypertension*).

**Bilan de la méthode 1** : **DIC-v4 est figée** (F1 VALIDATION 0,885). Il reste des mentions hors de tout lexique
(expressions descriptives comme *decreased renal function*, sigles non définis). 

Pour celles-ci, il faut une méthode qui reconnaît une mention à partir de son contexte, même sans l'avoir vue : c'est la méthode apprise.

## 5. Méthode 2 : un modèle de langue ajusté (BERT)

### 5.1 Le principe

On part d'un modèle **BERT pré-entraîné** sur des millions de textes biomédicaux (interprétation du sens des mots selon
leur contexte en devinant les mots masqués). 

On ajuste ensuite le modèle téléchargé sur TRAIN en ajoutant une couche finale qui apprend à choisir une étiquette BIO (`O, B-Chemical, I-Chemical, B-Disease, I-Disease`) pour chaque mot. 

Les résumés sont découpés en fenêtres de phrases entières et les mentions sont reconstruites à partir des étiquettes prédites (`src/apprise.py`). 

Les entraînements ont été faits sur Google Colab (GPU T4) ; les courbes ci-dessous sont celles enregistrées pendant ces entraînements.

### 5.2 APP-v1 : BioBERT avec les paramètres du TD

Modèle `dmis-lab/biobert-base-cased-v1.1`, paramètres du TD (taux 2e-5, lots de 8, 3 époques). 

Nous avons testé deux réglages sur VALIDATION : le nombre d'époques et un **poids plus fort sur les mots d'entité** (10 au lieu de 1), pour augmenter
le rappel :

In [17]:
pd.read_csv(HISTORIQUE / "apprise_reglage_validation.csv").drop(columns="temps_prediction_validation_s").sort_values("F1", ascending=False)

,poids,epoques,P,R,F1,F1 Chemical,F1 Disease,temps_entrainement_s
4,1.0,2,0.8533,0.8558,0.8546,0.9128,0.7813,84.6
5,1.0,3,0.8263,0.8826,0.8535,0.9104,0.7842,84.6
1,10.0,2,0.7182,0.8956,0.7971,0.8665,0.7168,81.9
2,10.0,3,0.6852,0.8855,0.7726,0.8499,0.6858,81.9
3,1.0,1,0.6849,0.8004,0.7382,0.8157,0.6480,84.6
0,10.0,1,0.5879,0.8624,0.6992,0.7894,0.5983,81.9


Note : Le poids 10 augmente bien le rappel mais fait chuter la précision, on ne le retient donc pas. 

Configuration choisie : poids 1, 2 époques, **F1 0,855**, soit déjà 6 points de plus que le meilleur dictionnaire.

### 5.3 Entraînement du modèle suffisant ? (APP-v2)

En relisant les chiffres, nous avons remarqué que 2 époques ne représentaient que 140 pas d'apprentissage et que le
F1 montait encore fortement d'une époque à l'autre (0,74 → 0,85) : le modèle semblait **sous-entraîné**. 

Nous avons
donc relancé avec 10 époques évaluées une à une, une montée progressive du taux (warmup 10 %), deux taux et deux
graines (pour mesurer ce qui relève du hasard). 

Règle de choix : On choisit la configuration avec la meilleure moyenne des deux
graines. Si une autre config présente une moyenne à moins de 0,005 du meilleur atteinte en moins d'époques, on la sélectionne elle.

In [18]:
v2 = pd.read_csv(HISTORIQUE / "apprise_v2_reglage_validation.csv")
courbes = v2.groupby(["taux", "epoques"]).F1.mean().unstack().round(3)
courbes.index = [f"taux {t:g}" for t in courbes.index]
display(courbes)
ecart = v2.groupby(["taux", "epoques"]).F1.agg(lambda s: s.max() - s.min()).max()
print(f"Écart maximal entre les deux graines : {ecart:.3f}")

epoques,1,2,3,4,5,6,7,8,9,10
taux 3e-05,0.747,0.866,0.878,0.881,0.884,0.882,0.887,0.885,0.889,0.887
taux 5e-05,0.835,0.879,0.883,0.892,0.892,0.890,0.893,0.895,0.895,0.896


Écart maximal entre les deux graines : 0.013


La configuration choisie est celle avec le taux de 5e-5, le F1 moyen atteint 0,892 dès **4 époques** (choix de la règle),
puis plafonne 
On estime qu'au-delà, les différences sont de l'ordre de l'écart entre graines (bruit). 

APP-v2 gagne +3,8 points (0,855 → **0,893**), surtout en Disease. 
Le gain vient alors de l'entraînement et non du modèle.

### 5.4 Un modèle plus spécialisé ? (APP-v3)

Nous avons voulu poursuivre en vérifiant si un autre modèle pré-entraîné ferait mieux avec le même entraînement. 
Nous avons comparé BioBERT à **PubMedBERT** et **BioLinkBERT**, appris directement sur PubMed avec leur propre vocabulaire. 
Leur vocabulaire découpe beaucoup moins les termes médicaux. 

Par exemple, BioBERT découpe « Doxorubicin-induced cardiomyopathy » en 12 sous-mots : *do _ x _ or _ ubi _ cin _ - _ induced _ card _ io _ my _ op _ athy*, contre trois  seulement pour PubMedBERT :

In [19]:
from transformers import BertTokenizerFast, AutoModelForTokenClassification
MODELE_V3 = RACINE / "modeles" / "apprise_v3"
tok_v3 = BertTokenizerFast.from_pretrained(MODELE_V3, do_lower_case=True)
print("PubMedBERT :", tok_v3.tokenize("Doxorubicin-induced cardiomyopathy"))

PubMedBERT : ['doxorubicin', '-', 'induced', 'cardiomyopathy']


Même paramètres d'entraînement que APP-v2 (taux 5e-5, 10 époques, deux graines) ; règle fixée à l'avance : remplacer APP-v2 si le
gain moyen dépasse 0,005.

In [20]:
v3 = pd.read_csv(HISTORIQUE / "apprise_v3_reglage_validation.csv")
courbes3 = pd.concat([v2[v2.taux == 5e-5].assign(modele="BioBERT (APP-v2)"), v3]).groupby(["modele", "epoques"]).F1.mean().unstack().round(3)
courbes3 

epoques,1,2,3,4,5,6,7,8,9,10
modele,,,,,,,,,,
BioBERT (APP-v2),0.835,0.879,0.883,0.892,0.892,0.890,0.893,0.895,0.895,0.896
BioLinkBERT,0.793,0.881,0.894,0.901,0.906,0.906,0.902,0.908,0.906,0.907
PubMedBERT,0.795,0.875,0.892,0.898,0.902,0.905,0.911,0.908,0.909,0.909


À partir de 3 époques, les deux nouveaux modèles dépassent BioBERT à **chaque** époque.

**PubMedBERT à 7 époques** est retenu (F1 moyen 0,911, +0,019 par rapport à BioBERT). L'écart avec BioLinkBERT
(0,006) est proche de la variation entre graines : les deux se valent donc. 

### 5.5 Prédictions du modèle figé APP-v3

Le modèle est rechargé et prédit toute la partition VALIDATION (sur CPU, ≈ 6 min ; ≈ 15 s sur GPU). On utilise la
version « rapide » du même tokeniseur (`BertTokenizerFast`, mêmes découpages), nécessaire pour retrouver à quel mot
appartient chaque sous-mot.

In [21]:
import torch
from src.apprise import construire_fenetres, predire

APPAREIL = "cuda" if torch.cuda.is_available() else "cpu"
modele_v3 = AutoModelForTokenClassification.from_pretrained(MODELE_V3).to(APPAREIL)
fenetres = {p: construire_fenetres([bio[d.document_id] for d in parts[p]], textes, tok_v3, 512) for p in ("validation", "test")}
t0 = time.perf_counter()
pred_app_val = predire(modele_v3, tok_v3, fenetres["validation"], bio, 512)
duree_val = time.perf_counter() - t0
print(f"{len(fenetres['validation'])} fenêtres, {len(pred_app_val)} mentions prédites en {duree_val:.0f} s ({APPAREIL})")
pd.DataFrame(tableau_scores(ref["validation"], pred_app_val, "APP-v3", "validation")).set_index("categorie")[["VP", "FP", "FN", "precision", "rappel", "F1"]]

515 fenêtres, 9756 mentions prédites en 376 s (cpu)


,VP,FP,FN,precision,rappel,F1
categorie,,,,,,
Chemical,5141,299,206,0.9450,0.9615,0.9532
Disease,3680,636,564,0.8526,0.8671,0.8598
global (micro),8821,935,770,0.9042,0.9197,0.9119


**Réentraîner APP-v3 (optionnel)** : la cellule suivante refait l'entraînement retenu (PubMedBERT, taux 5e-5,
warmup 10 %, graine 42, 7 époques) avec `src.apprise.entrainer`, puis remplace le modèle utilisé ci-dessus. Elle ne
s'exécute que si `REENTRAINER = True` (long sur CPU).

In [22]:
REENTRAINER = False
if REENTRAINER:
    from src.apprise import encoder, entrainer
    from transformers import BertTokenizerFast
    nom = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
    tok = BertTokenizerFast.from_pretrained(nom, do_lower_case=True)
    fen_train = construire_fenetres([bio[d.document_id] for d in parts["train"]], textes, tok, 512)
    donnees = encoder(fen_train, bio, tok, 512)[0]
    points, duree = entrainer(nom, donnees, tok, RACINE / "modeles" / "checkpoints_v3", taux=5e-5, epoques=7, graine=42)
    print(f"Entraînement : {duree / 60:.1f} min")
    tok_v3, modele_v3 = tok, AutoModelForTokenClassification.from_pretrained(points[-1]).to(APPAREIL)
    fenetres = {p: construire_fenetres([bio[d.document_id] for d in parts[p]], textes, tok_v3, 512) for p in ("validation", "test")}
    pred_app_val = predire(modele_v3, tok_v3, fenetres["validation"], bio, 512)

## 6. Comparaison des versions sur VALIDATION

In [23]:
lignes = []
for nom, pred in {"DIC-v1": pred_v1, "DIC-v2": pred_v2, "DIC-v3": pred_v3, "DIC-v4": pred_v4, "APP-v3 (PubMedBERT)": pred_app_val}.items():
    e = erreurs(ref["validation"], pred)
    lignes.append({"version": nom, **scores(pred), **{k: len(e[k]) for k in ("correcte", "frontiere", "type", "manquee", "parasite")}})
v1_hist = pd.read_csv(HISTORIQUE / "apprise_reglage_validation.csv").query("poids == 1 and epoques == 2").iloc[0]
v2_hist = v2.query("taux == 5e-5 and graine == 42 and epoques == 4").iloc[0]
for nom, h in {"APP-v1 (BioBERT, TD)": v1_hist, "APP-v2 (BioBERT mieux entraîné)": v2_hist}.items():
    lignes.append({"version": nom, **{k: h[k] for k in ("P", "R", "F1", "F1 Chemical", "F1 Disease")}})
ordre = ["DIC-v1", "DIC-v2", "DIC-v3", "DIC-v4", "APP-v1 (BioBERT, TD)", "APP-v2 (BioBERT mieux entraîné)", "APP-v3 (PubMedBERT)"]
comparaison_val = pd.DataFrame(lignes).set_index("version").loc[ordre]
comparaison_val

,P,R,F1,F1 Chemical,F1 Disease,correcte,frontiere,type,manquee,parasite
version,,,,,,,,,,
DIC-v1,0.8835,0.6438,0.7449,0.7715,0.7127,6175.0,578.0,13.0,2825.0,223.0
DIC-v2,0.8889,0.7128,0.7912,0.8289,0.7434,6836.0,576.0,8.0,2171.0,270.0
DIC-v3,0.8811,0.7314,0.7993,0.8333,0.7569,7015.0,627.0,8.0,1941.0,312.0
DIC-v4,0.9032,0.8669,0.8847,0.9174,0.8424,8314.0,437.0,8.0,832.0,446.0
"APP-v1 (BioBERT, TD)",0.8533,0.8558,0.8546,0.9128,0.7813,NaN,NaN,NaN,NaN,NaN
APP-v2 (BioBERT mieux entraîné),0.8886,0.8973,0.8929,0.9358,0.8392,NaN,NaN,NaN,NaN,NaN
APP-v3 (PubMedBERT),0.9042,0.9197,0.9119,0.9532,0.8598,8821.0,432.0,4.0,334.0,499.0


Colonnes d'erreurs (`tals.erreurs`) : **frontière** = bonne mention mais bornes différentes ; **type** = bonnes
bornes, mauvais type ; **manquée** = aucune prédiction ne recouvre la mention ; **parasite** = prédiction qui ne
recouvre aucune mention.

**Lecture** :
- Les deux méthodes figées ont la **même précision** (0,90) : la différence se fait sur le **rappel** (0,87 pour
  DIC-v4, 0,92 pour APP-v3). Les mentions manquées passent de ≈ 1 950 (DIC-v3) à ≈ 830 (DIC-v4) et ≈ 330 (APP-v3).
- **Disease est la catégorie difficile** pour les deux méthodes (7 à 9 points sous Chemical) : les maladies sont
  souvent des expressions de plusieurs mots aux bornes discutables (« deterioration of vision »), alors que les
  produits chimiques ont des noms plus réguliers.
- Chaque version améliore la précédente : 
    - Dans la méthode par dictionnaire, le lexique (v3 → v4 : +8,5 points) a pesé bien plus que
  les règles
    - Dans la méthode apprise, les réglages de l'entraînement (v1 → v2 : +3,8 points) a pesé plus que le changement de modèle
  (v2 → v3 : +1,9).

**Les deux méthodes se complètent-elles ?**

In [24]:
justes_dic, justes_app = pred_v4 & ref["validation"], pred_app_val & ref["validation"]
dic_seul = justes_dic - justes_app
print(f"Trouvées par les deux : {len(justes_dic & justes_app)} ; APP-v3 seul : {len(justes_app - justes_dic)} ; "
      f"dictionnaire seul : {len(dic_seul)}")
print(f"Rappel APP-v3 : {len(justes_app) / len(ref['validation']):.3f} ; rappel de l'union : "
      f"{len(justes_app | justes_dic) / len(ref['validation']):.3f}")
print("Trouvées par le dictionnaire seul :", Counter(forme(m) for m in dic_seul).most_common(10))

Trouvées par les deux : 7992 ; APP-v3 seul : 829 ; dictionnaire seul : 322
Rappel APP-v3 : 0.920 ; rappel de l'union : 0.953
Trouvées par le dictionnaire seul : [('hepatitis B', 9), ('COCs', 7), ('cytotoxicity', 6), ('cibenzoline', 5), ('angiotensin', 5), ('depression', 5), ('MCAO', 5), ('hemolytic anemia', 5), ('serotonin', 5), ('tumor', 5)]


Avec MeSH, le dictionnaire et le modèle trouvent en grande partie les mêmes mentions (≈ 8 000 en commun). Le
dictionnaire en trouve encore ≈ 320 que le modèle rate, surtout des **sigles définis dans le texte** (règle de DIC-v2,
*COCs*, *MCAO*) et des formes que TRAIN annote régulièrement.

L'union des deux méthodes atteindrait un rappel de 0,95, mais une simple
union ajouterait aussi tous les faux positifs des deux méthodes. 
Une combinaison testée avec DIC-v3 ne dépassait pas APP-v3 seul. 
Refaire cet essai avec DIC-v4 reste une piste.

## 7. Évaluation sur TEST

Versions figées : **DIC-v4** et **APP-v3**, appliquées une fois aux 500 résumés de TEST, sans aucun réglage.

In [25]:
t0 = time.perf_counter()
pred_dic_test = dic_v4.extraire_corpus(parts["test"])
t_dic_test = time.perf_counter() - t0
t0 = time.perf_counter()
pred_app_test = predire(modele_v3, tok_v3, fenetres["test"], bio, 512)
t_app_test = time.perf_counter() - t0
pred_test = {"DIC-v4 (dictionnaire)": pred_dic_test, "APP-v3 (PubMedBERT)": pred_app_test}
scores_test = pd.concat([tableau_scores(ref["test"], p, nom, "test") for nom, p in pred_test.items()])
display(scores_test.drop(columns=["partition", "denominateur_nul", "temps_s"]).set_index(["methode", "categorie"]))

ecart = pd.DataFrame({"F1 VALIDATION": [comparaison_val.loc["DIC-v4", "F1"], comparaison_val.loc["APP-v3 (PubMedBERT)", "F1"]],
                      "F1 TEST": scores_test[scores_test.categorie == "global (micro)"].F1.values}, index=list(pred_test))
ecart["écart"] = (ecart["F1 TEST"] - ecart["F1 VALIDATION"]).round(4)

# Sorties au format du guide : une ligne par (méthode, document, catégorie)
with open(SORTIES / "predictions_extraction.jsonl", "w", encoding="utf-8") as f:
    for methode, p in (("dictionnaire", pred_dic_test), ("apprise", pred_app_test)):
        ecrire_predictions(p, methode, parts["test"], SORTIES / "_tmp.jsonl")
        f.write((SORTIES / "_tmp.jsonl").read_text(encoding="utf-8"))
(SORTIES / "_tmp.jsonl").unlink()
ecart

VP    FP    FN  precision  rappel  \
methode               categorie                                             
DIC-v4 (dictionnaire) Chemical        4956   396   429     0.9260  0.9203   
                      Disease         3649   549   775     0.8692  0.8248   
                      global (micro)  8605   945  1204     0.9010  0.8773   
APP-v3 (PubMedBERT)   Chemical        5085   486   300     0.9128  0.9443   
                      Disease         3843   749   581     0.8369  0.8687   
                      global (micro)  8928  1235   881     0.8785  0.9102   

                                          F1  
methode               categorie               
DIC-v4 (dictionnaire) Chemical        0.9232  
                      Disease         0.8464  
                      global (micro)  0.8890  
APP-v3 (PubMedBERT)   Chemical        0.9283  
                      Disease         0.8525  
                      global (micro)  0.8941

,F1 VALIDATION,F1 TEST,écart
DIC-v4 (dictionnaire),0.8847,0.8890,0.0043
APP-v3 (PubMedBERT),0.9119,0.8941,-0.0178


**Lecture** :
- **APP-v3 ne dépasse plus DIC-v4 que de 0,5 point** de F1 sur TEST (0,894 contre 0,889).
 Le dictionnaire est plus **précis** (0,90 contre 0,88), le modèle a un meilleur **rappel** (0,91 contre 0,88).

  En Chemical, les deux comparables (0,923 contre 0,928) ; en Disease, l'écart est de 0,6 point.
  
- On note que **Le dictionnaire est stable** de VALIDATION à TEST (+0,004) : choisir la variante MeSH sur VALIDATION n'a pas crée d'écart optimiste (on peut considérer le choix justifié).

- **APP-v3 perd ≈ 1,8 point**, surtout par la précision en Chemical : c'est l'optimisme attendu,
  car le modèle, l'époque et la graine ont été choisis sur VALIDATION.

- Sans MeSH (DIC-v3, évaluée sur TEST le 30/10), le dictionnaire obtenait 0,798 : le lexique apporte +9 points sur TEST.

In [26]:
def lire_outil(chemin, categorie):
    # Fichier PubTator de prédictions : document, début, fin, texte, type[, identifiant]
    sortie = set()
    for ligne in open(chemin, encoding="utf-8"):
        c = ligne.rstrip("\n").split("\t")
        if len(c) >= 5 and c[4] == categorie:
            m = Mention(c[0], int(c[1]), int(c[2]), categorie)
            assert textes[m.doc][m.debut:m.fin] == c[3]   # positions vérifiées sur le texte de référence
            sortie.add(m)
    return sortie

officiels = (lire_outil(CDR / "tmChem.TestSet" / "TestSet.tmChem.PubTator.txt", "Chemical")
             | lire_outil(CDR / "DNorm.TestSet" / "TestSet.DNorm.PubTator.txt", "Disease"))
etat_art = pd.concat([scores_test, tableau_scores(ref["test"], officiels, "tmChem + DNorm (outils officiels)", "test")])
etat_art.pivot(index="methode", columns="categorie", values="F1")[["Chemical", "Disease", "global (micro)"]]

categorie,Chemical,Disease,global (micro)
methode,,,
APP-v3 (PubMedBERT),0.9283,0.8525,0.8941
DIC-v4 (dictionnaire),0.9232,0.8464,0.8890
tmChem + DNorm (outils officiels),0.9220,0.8024,0.8678


| Modèle publié (TEST BC5CDR, F1 entité) | Chemical | Disease | Source |
|---|---|---|---|
| BERT-base (domaine général) | 89,3 | 81,4 | Gu et al. 2021 |
| BioBERT | 92,9 | 84,7 | Gu et al. 2021 (BLURB) |
| **PubMedBERT** (même modèle qu'APP-v3) | **93,3** | **85,6** | Gu et al. 2021 (BLURB) |
| BioLinkBERT-base | 93,8 | 86,1 | Yasunaga et al. 2022 |
| GPT-4 sans ajustement (0 / 1 exemple) | 79,9 / 83,3 | — | Chen et al. 2023-2025 |

- **APP-v3 est au niveau publié pour son modèle** (à moins d'un point), avec un protocole plus contraint : un seul
  modèle pour les deux types (les publications en entraînent souvent un par type).
- **Avec MeSH, notre dictionnaire dépasse les outils officiels** de 2,1 points : à égalité en Chemical (0,923 contre
  0,922), +4,4 points en Disease (0,846 contre 0,802). Ces outils utilisaient eux aussi des lexiques ; notre filtrage
  appris sur TRAIN adapte le lexique aux conventions d'annotation de BC5CDR. Sans MeSH (DIC-v3), nous étions 7 points
  en dessous.

  Nous en concluons que l'écart avec les outils officiels venait bien du vocabulaire.

- L'écart avec les grands modèles, comme GPT-4, vient certainement de l'absence d'ajustement et donc du manque de spécificité sur notre corpus.
   
- **Limites** : En nous renseignant un peu plus, nous avons noté que MeSH est aussi la ressource qui sert à normaliser les annotations de BC5CDR, de ce fait, il se pourrait que sa version 2026 puisse
  contenir des noms de substances ajoutés lors de l'indexation des résumés du corpus eux-mêmes.
  
- PubMedBERT dépasse les outils officiels de ≈ 2,6 points, surtout en Disease. 

In [27]:
err_app = erreurs(ref["validation"], pred_app_val)
display(pd.DataFrame({cat: {"frontière": sum(a.type == cat for a, _ in err_app["frontiere"]),
                            "type": sum(a.type == cat for a, _ in err_app["type"]),
                            "manquée": sum(m.type == cat for m in err_app["manquee"]),
                            "parasite": sum(m.type == cat for m in err_app["parasite"])} for cat in CATEGORIES}).T)
longues = sum(p.longueur > a.longueur for a, p in err_app["frontiere"])
print(f"Frontières : prédiction plus longue que la référence dans {longues} cas, plus courte dans "
      f"{len(err_app['frontiere']) - longues}")
manq_hors = sum(forme(m).lower() not in formes_train for m in err_app["manquee"])
print(f"Manquées : {manq_hors} sur {len(err_app['manquee'])} ont une forme absente de TRAIN")

,frontière,type,manquée,parasite
Chemical,89,1,116,199
Disease,343,3,218,300


Frontières : prédiction plus longue que la référence dans 244 cas, plus courte dans 188
Manquées : 251 sur 334 ont une forme absente de TRAIN


On remarque une proportion importante d'erreurs dans **Disease** (frontières, manquées et parasites).
Les frontières se trompent dans les deux sens, et les mentions manquées sont surtout des formes jamais vues.

Un tirage d'erreurs de chaque sorte, avec leur contexte :

In [28]:
def contexte(m, large=45):
    t = textes[m.doc]
    return f"…{t[max(0, m.debut - large):m.debut]}[{t[m.debut:m.fin]}]{t[m.fin:m.fin + large]}…"

hasard = random.Random(42)
exemples = []
for a, p in hasard.sample(err_app["frontiere"], 3):
    exemples.append({"sorte": "frontière", "référence": f"{forme(a)} ({a.type})", "prédit": f"{forme(p)} ({p.type})", "contexte": contexte(a)})
for m in hasard.sample(err_app["manquee"], 3):
    exemples.append({"sorte": "manquée", "référence": f"{forme(m)} ({m.type})", "prédit": "—", "contexte": contexte(m)})
for m in hasard.sample(err_app["parasite"], 3):
    exemples.append({"sorte": "parasite", "référence": "—", "prédit": f"{forme(m)} ({m.type})", "contexte": contexte(m)})
pd.DataFrame(exemples)

,sorte,référence,prédit,contexte
0,frontière,calcium carbon-ate (Chemical),calcium carbon-ate-sodium bicarbonate (Chemical),… who had regularly consumed large amounts of [calcium carbon-ate]-sodium bicarbonate powders for more than 20 …
1,frontière,refractory seizures (Disease),seizures (Disease),…escribed because of its efficacy in treating [refractory seizures]. Urologists should be aware that this medica…
2,frontière,angiotensin (Chemical),angiotensin-converting enzyme inhibitor (Chemical),…nction in patients with heart failure during [angiotensin]-converting enzyme inhibitor therapy: results…
3,manquée,mental slowing (Disease),—,…rly was associated with increased subjective [mental slowing] after trihexyphenidyl anticholinergic challe…
4,manquée,MCAO (Disease),—,"…g beginning 2 h after MCAO. Four hours after [MCAO], the rats were killed and the brains harvest…"
5,manquée,QTP (Chemical),—,"…th a mood stabilizer (lithium or divalproex, [QTP] + Li/DVP) (n = 196) compared to placebo and …"
6,parasite,—,heart valves (Disease),…terature on LMWH in patients with mechanical [heart valves] was extensively reviewed. RESULTS: There wer…
7,parasite,—,obstruction (Disease),…ration of the oropharynx that predisposes to [obstruction] during sleep. Development of the obstructive…
8,parasite,—,neurosteroids (Chemical),"…t as one of the most potent memory-enhancing [neurosteroids] to be examined in rodent learning studies, w…"


**Cinq cas commentés**, choisis pour couvrir des causes différentes (sorties des deux méthodes autour de la mention) :

In [29]:
CAS = [   # (document, forme, ce qu'il illustre)
    ("1628552", "deterioration of vision", "frontière : maladie décrite par une expression courante"),
    ("15867025", "hepatitis B and rubella", "coordination et contexte de vaccination"),
    ("12851669", "COCs", "sigle défini dans le texte, jamais annoté dans TRAIN"),
    ("3088349", "contralateral rotation", "symptôme d'expérimentation animale, jamais vu"),
    ("8800187", "antidepressant", "annotation irrégulière entre TRAIN et VALIDATION"),
]
lignes = []
for doc, f, sujet in CAS:
    t = textes[doc]
    i = t.find(f)
    zone = lambda ms: sorted({f"{t[m.debut:m.fin]} ({m.type[:4]})" for m in ms if m.doc == doc and m.debut < i + len(f) + 15 and m.fin > i - 15})
    lignes.append({"cas": sujet, "contexte": t[max(0, i - 50):i + len(f) + 30],
                   "référence": zone(ref["validation"]), "APP-v3": zone(pred_app_val), "DIC-v4": zone(pred_v4)})
display(pd.DataFrame(lignes))
annot_train = sum(m.texte.lower() == "antidepressant" for d in parts["train"] for m in d.mentions)
annot_val = sum(m.texte.lower() == "antidepressant" for d in parts["validation"] for m in d.mentions)
occ = {p: sum(len(re.findall(r"\bantidepressant\b", d.texte, re.I)) for d in parts[p]) for p in ("train", "validation")}
print(f"« antidepressant » : annoté {annot_train} fois sur {occ['train']} dans TRAIN, {annot_val} fois sur {occ['validation']} dans VALIDATION")

,cas,contexte,référence,APP-v3,DIC-v4
0,frontière : maladie décrite par une expression courante,"chloroquine ingestion presented with progressive deterioration of vision, easy fatiguability, dyspnoea","[deterioration of vision (Dise), fatiguability (Dise)]",[of vision (Dise)],[]
1,coordination et contexte de vaccination,Assessment of perinatal hepatitis B and rubella prevention in New Hampshire d,"[hepatitis B (Dise), rubella (Dise)]",[hepatitis B and rubella (Dise)],"[hepatitis B (Dise), rubella (Dise)]"
2,"sigle défini dans le texte, jamais annoté dans TRAIN",and among women on combined oral contraceptives (COCs). METHODS: From the Danish Na,"[COCs (Chem), combined oral contraceptives (Chem)]",[oral contraceptives (Chem)],"[COCs (Chem), oral contraceptives (Chem)]"
3,"symptôme d'expérimentation animale, jamais vu",Transient contralateral rotation following unilateral substant,[contralateral rotation (Dise)],[],[]
4,annotation irrégulière entre TRAIN et VALIDATION,is a major contributor to mortality in tricyclic antidepressant overdose. Recent data suggest,[overdose (Dise)],"[antidepressant (Chem), overdose (Dise)]",[overdose (Dise)]


« antidepressant » : annoté 15 fois sur 15 dans TRAIN, 0 fois sur 11 dans VALIDATION


1. **« deterioration of vision »** : le modèle choisit une étiquette par mot ; il rate le début et ouvre la mention sur
   « of ». Avec le score strict, l'erreur compte deux fois (un faux positif, un faux négatif).


2. **« hepatitis B and rubella »** : la référence sépare les deux maladies, le modèle les fusionne (comme une mention
   composite). Le dictionnaire a raison ici en ignorant le contexte.

   
4. **« COCs »** : le sigle apparaît dans TRAIN sans y être annoté, donc le modèle a appris à l'ignorer ; le dictionnaire le retrouve grâce à la règle des sigles définis (« combined oral contraceptives (COCs) »).


5. **« contralateral rotation »** : chez le rat, un comportement anormal annoté comme Disease, avec des mots ordinaires jamais vus ; aucune des deux méthodes ne peux le deviner.


6. **« antidepressant »** : la mention est annotée Chemical à chacune de ses apparitions dans TRAIN et à aucune dans VALIDATION. puisque le modèle applique ce qu'il apprend sur TRAIN, l'erreur vient de l'**annotation**.

## 9. Limites, coûts et conclusion

**Coûts mesurés ici**

In [30]:
pd.DataFrame([
    {"méthode": "DIC-v4 (dictionnaire + MeSH + règles)", "matériel": "CPU",
     "construction": f"{t_construction_v4:.0f} s (TRAIN + MeSH ; + 1 min au premier téléchargement)",
     "extraction de 500 documents": f"{t_extraction_v4:.0f} s (VALIDATION), {t_dic_test:.0f} s (TEST)"},
    {"méthode": "APP-v3 (PubMedBERT ajusté)", "matériel": "GPU T4 pour entraîner",
     "construction": "≈ 3-4,5 min par entraînement de 10 époques ; 10 entraînements de APP-v1 à APP-v3",
     "extraction de 500 documents": f"{duree_val:.0f} s (VALIDATION), {t_app_test:.0f} s (TEST) sur {APPAREIL} ; ≈ 16 s sur GPU"},
]).set_index("méthode")

,matériel,construction,extraction de 500 documents
méthode,,,
DIC-v4 (dictionnaire + MeSH + règles),CPU,25 s (TRAIN + MeSH ; + 1 min au premier téléchargement),"3 s (VALIDATION), 3 s (TEST)"
APP-v3 (PubMedBERT ajusté),GPU T4 pour entraîner,"≈ 3-4,5 min par entraînement de 10 époques ; 10 entraînements de APP-v1 à APP-v3","376 s (VALIDATION), 379 s (TEST) sur cpu ; ≈ 16 s sur GPU"


**Limites**
- Frontière décalée d'un mot compte comme deux erreurs, alors que la mention est souvent utile (« vision » au lieu de « deterioration of vision »).
  
- DIC-v4 dépend d'un lexique externe volumineux (≈ 840 000 termes, ≈ 1,3 Go de mémoire à la construction) ; ne reconnaît toujours pas les descriptions absentes de tout lexique (*decreased renal function*).
- Certaines erreurs viennent d'un problème d'irrégularité dans l'annotation (erreur 5).

  
- Les mentions composites ne sont pas bien découpées, certaines pas représentables en BIO.

**Conclusion**

On a suivi un cheminement en testant plusieurs versions successives, tout en visant de meilleurs résultats : le dictionnaire était précis mais manquait des mentions ;
nos règles en ont retrouvé une partie, jusqu'à constater que l'essentiel du reste était du vocabulaire jamais vu dans
TRAIN. 

Réponse à cette limite : **élargir le vocabulaire** avec MeSH, filtré sur TRAIN pour respecter les
conventions d'annotation (DIC-v4), ou **apprendre à généraliser** avec un modèle. 

Pour le modèle, corriger l'entraînement a plus apporté que changer de modèle, et un modèle avec un vocabulaire biomédical plus précis a donné de meilleurs gains.

| Version figée | F1 VALIDATION | F1 TEST | Points forts | Points faibles |
|---|---|---|---|---|
| DIC-v4 | 0,885 | 0,889 | précis, stable, rapide, sans GPU, explicable ; dépasse les outils officiels | dépend d'un lexique externe ; ne reconnaît pas les formes absentes de TRAIN et de MeSH |
| APP-v3 | 0,912 | 0,894 | meilleur rappel, au niveau des publications, sans lexique | GPU pour entraîner, erreurs moins explicables, sensible aux irrégularités d'annotation |
